# 04 - Modélisation avec Choix Justifiés

## Objectif
1. Implémenter 4 modèles de base
2. Comparer 2 stratégies d'entraînement (pooled vs separate+ensemble)
3. Analyser l'importance des caractéristiques avec 4 méthodes
4. Tests de signification statistique

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.abspath('..'))
from src.data_processing import (
    load_titanic, load_lusitania, load_sewol,
    clean_titanic, clean_lusitania, clean_sewol,
    create_unified_dataset
)
from src.feature_engineering import (
    full_feature_engineering_pipeline, prepare_features_for_modeling
)
from src.models import (
    get_base_models, evaluate_model, cross_validate_models,
    compute_learning_curves, train_pooled_strategy,
    train_separate_strategy, compute_permutation_importance,
    compute_drop_column_importance, get_logistic_coefficients,
    mcnemar_test
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 6)

print('Modules chargés.')

In [ ]:
# Préparer les données
titanic_clean = clean_titanic(load_titanic())
lusitania_clean = clean_lusitania(load_lusitania())
sewol_clean = clean_sewol(load_sewol())
unified = create_unified_dataset(titanic_clean, lusitania_clean, sewol_clean)

# Feature engineering complet
df_enriched = full_feature_engineering_pipeline(unified)

# Séparer : Titanic + Lusitania pour l'entraînement, Sewol pour le test holdout
train_mask = df_enriched['disaster'].isin(['titanic', 'lusitania'])
test_mask = df_enriched['disaster'] == 'sewol'

df_train = df_enriched[train_mask].copy()
df_test = df_enriched[test_mask].copy()

print(f"Train (Titanic+Lusitania): {len(df_train)} échantillons")
print(f"Test holdout (Sewol): {len(df_test)} échantillons")

# Préparer les features
X_train, y_train, scaler, feature_names = prepare_features_for_modeling(df_train, scale=True)

# Appliquer le même scaler au test
X_test_raw, y_test, _, _ = prepare_features_for_modeling(df_test, scale=False)
X_test = pd.DataFrame(
    scaler.transform(X_test_raw),
    columns=feature_names,
    index=X_test_raw.index
)

disaster_train = df_train['disaster'].values
disaster_test = df_test['disaster'].values

print(f"\nFeatures: {len(feature_names)}")
print(f"Train y distribution: {y_train.value_counts().to_dict()}")
print(f"Test y distribution: {y_test.value_counts().to_dict()}")

---
## 1. Modèles de Base - Validation Croisée

### Justification des choix de modèles :
1. **Régression Logistique** : Baseline interprétable. Les coefficients donnent directement l'effet de chaque feature sur la probabilité de survie. Sensible aux features corrélées.
2. **Random Forest** : Robuste aux outliers et aux features non-informatives. Gère naturellement les non-linéarités et les interactions.
3. **XGBoost** : État de l'art pour les données tabulaires. Gère les valeurs manquantes nativement et optimise le biais-variance via le boosting.
4. **SVM (RBF)** : Frontières de décision flexibles, bon en haute dimension. Nécessite une normalisation préalable.

In [ ]:
# Obtenir les modèles de base
models = get_base_models()

# Validation croisée 5-fold
print("Validation croisée 5-fold sur Titanic + Lusitania...")
cv_results = cross_validate_models(models, X_train, y_train, cv=5)

print("\n=== Résultats de la Validation Croisée ===")
display_cols = ['accuracy_mean', 'accuracy_std', 'f1_mean', 'f1_std', 'roc_auc_mean', 'roc_auc_std']
print(cv_results[display_cols].round(4).to_string())

In [ ]:
# Visualisation des résultats CV
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Comparaison des 4 Modèles (Validation Croisée 5-Fold)', fontsize=14, fontweight='bold')

metrics_to_plot = ['accuracy', 'f1', 'roc_auc']
titles = ['Accuracy', 'F1 Score', 'ROC AUC']
colors = ['steelblue', 'mediumseagreen', 'coral', 'mediumpurple']

for i, (metric, title) in enumerate(zip(metrics_to_plot, titles)):
    means = cv_results[f'{metric}_mean']
    stds = cv_results[f'{metric}_std']
    bars = axes[i].bar(range(len(means)), means, yerr=stds, color=colors,
                       capsize=5, edgecolor='black', linewidth=0.5)
    axes[i].set_title(title)
    axes[i].set_xticks(range(len(means)))
    axes[i].set_xticklabels(means.index, rotation=15, ha='right')
    axes[i].set_ylim(0.5, 1.0)
    # Annoter les valeurs
    for j, (m, s) in enumerate(zip(means, stds)):
        axes[i].annotate(f'{m:.3f}', xy=(j, m + s + 0.01), ha='center', fontsize=9)

plt.tight_layout()
plt.show()

---
## 2. Courbes d'Apprentissage

In [ ]:
print("Calcul des courbes d'apprentissage...")
curves = compute_learning_curves(models, X_train, y_train)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Courbes d\'Apprentissage', fontsize=14, fontweight='bold')

for idx, (name, data) in enumerate(curves.items()):
    ax = axes[idx // 2, idx % 2]
    ax.fill_between(data['train_sizes'],
                    data['train_mean'] - data['train_std'],
                    data['train_mean'] + data['train_std'],
                    alpha=0.1, color='blue')
    ax.fill_between(data['train_sizes'],
                    data['val_mean'] - data['val_std'],
                    data['val_mean'] + data['val_std'],
                    alpha=0.1, color='orange')
    ax.plot(data['train_sizes'], data['train_mean'], 'o-', color='blue',
            label='Score entraînement')
    ax.plot(data['train_sizes'], data['val_mean'], 'o-', color='orange',
            label='Score validation')
    ax.set_title(name)
    ax.set_xlabel('Taille de l\'entraînement')
    ax.set_ylabel('Accuracy')
    ax.legend(loc='lower right')
    ax.set_ylim(0.5, 1.05)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 3. Stratégie 1 : Entraînement Regroupé (Pooled)

### Principe
Entraîner un seul modèle sur toutes les catastrophes d'entraînement (Titanic + Lusitania), puis tester sur le Sewol.

### Quand cette stratégie excelle :
- Les catastrophes partagent des patterns communs (ex: l'effet du genre)
- On a besoin de plus de données pour des modèles complexes

### Quand cette stratégie échoue :
- Les relations features→survie sont très différentes entre catastrophes
- Le modèle est biaisé vers la catastrophe la plus représentée

In [ ]:
print("=== Stratégie POOLED ===")
pooled_results, pooled_models = train_pooled_strategy(
    get_base_models(), X_train, y_train, X_test, y_test
)

pooled_df = pd.DataFrame({
    name: {k: v for k, v in metrics.items() if k != 'confusion_matrix'}
    for name, metrics in pooled_results.items()
}).T

print("\nPerformance sur le Sewol (catastrophe réservée) :")
print(pooled_df.round(4).to_string())

---
## 4. Stratégie 2 : Entraînement Séparé + Ensemble

### Principe
Entraîner un modèle par catastrophe (Titanic et Lusitania séparément), puis combiner les prédictions par vote moyen.

### Quand cette stratégie excelle :
- Chaque catastrophe a des patterns distincts
- On veut des modèles spécialisés

### Quand cette stratégie échoue :
- Peu de données par catastrophe
- Les catastrophes partagent des patterns qui ne sont appris que conjointement

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC

print("=== Stratégie SEPARATE + ENSEMBLE ===")

model_factories = {
    'Logistic Regression': lambda: LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': lambda: RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42),
    'XGBoost': lambda: XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, verbosity=0, use_label_encoder=False, eval_metric='logloss'),
    'SVM': lambda: SVC(kernel='rbf', probability=True, random_state=42)
}

separate_results = {}
for name, factory in model_factories.items():
    metrics, _ = train_separate_strategy(
        factory, X_train, y_train, disaster_train,
        X_test, y_test, disaster_test
    )
    separate_results[name] = {k: v for k, v in metrics.items() if k != 'confusion_matrix'}

separate_df = pd.DataFrame(separate_results).T

print("\nPerformance sur le Sewol (catastrophe réservée) :")
print(separate_df.round(4).to_string())

---
## 5. Comparaison des Stratégies

In [ ]:
# Comparaison côte à côte
print("=== COMPARAISON POOLED vs SEPARATE ===")

comparison = pd.DataFrame()
for model_name in pooled_df.index:
    comparison.loc[model_name, 'Pooled Accuracy'] = pooled_df.loc[model_name, 'accuracy']
    comparison.loc[model_name, 'Separate Accuracy'] = separate_df.loc[model_name, 'accuracy']
    comparison.loc[model_name, 'Pooled F1'] = pooled_df.loc[model_name, 'f1']
    comparison.loc[model_name, 'Separate F1'] = separate_df.loc[model_name, 'f1']
    if 'roc_auc' in pooled_df.columns and 'roc_auc' in separate_df.columns:
        comparison.loc[model_name, 'Pooled AUC'] = pooled_df.loc[model_name, 'roc_auc']
        comparison.loc[model_name, 'Separate AUC'] = separate_df.loc[model_name, 'roc_auc']

print(comparison.round(4).to_string())

In [ ]:
# Visualisation comparative
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Comparaison des Stratégies d\'Entraînement', fontsize=14, fontweight='bold')

x = np.arange(len(pooled_df.index))
width = 0.35

# Accuracy
axes[0].bar(x - width/2, pooled_df['accuracy'], width, label='Pooled', color='steelblue')
axes[0].bar(x + width/2, separate_df['accuracy'], width, label='Separate+Ensemble', color='coral')
axes[0].set_title('Accuracy sur Sewol (Holdout)')
axes[0].set_xticks(x)
axes[0].set_xticklabels(pooled_df.index, rotation=15, ha='right')
axes[0].legend()
axes[0].set_ylim(0, 1)

# F1
axes[1].bar(x - width/2, pooled_df['f1'], width, label='Pooled', color='steelblue')
axes[1].bar(x + width/2, separate_df['f1'], width, label='Separate+Ensemble', color='coral')
axes[1].set_title('F1 Score sur Sewol (Holdout)')
axes[1].set_xticks(x)
axes[1].set_xticklabels(pooled_df.index, rotation=15, ha='right')
axes[1].legend()
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

---
## 6. Tests de Signification Statistique

In [ ]:
from scipy.stats import ttest_rel
from sklearn.model_selection import StratifiedKFold

# Paired t-test sur les scores de validation croisée
print("=== Tests de Signification Statistique ===")
print("\n--- Paired t-test entre modèles (5-fold CV) ---")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
all_cv_scores = {}

for name, model in get_base_models().items():
    scores = []
    for train_idx, val_idx in skf.split(X_train, y_train):
        model_copy = type(model)(**model.get_params())
        model_copy.fit(X_train.iloc[train_idx], y_train.iloc[train_idx])
        scores.append(model_copy.score(X_train.iloc[val_idx], y_train.iloc[val_idx]))
    all_cv_scores[name] = scores

# Comparer chaque paire de modèles
model_names = list(all_cv_scores.keys())
for i in range(len(model_names)):
    for j in range(i+1, len(model_names)):
        t_stat, p_val = ttest_rel(all_cv_scores[model_names[i]], all_cv_scores[model_names[j]])
        sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
        print(f"{model_names[i]} vs {model_names[j]}: t={t_stat:.4f}, p={p_val:.4f} [{sig}]")

In [ ]:
# Test de McNemar entre stratégies (Pooled vs Separate) pour chaque modèle
print("\n--- Test de McNemar : Pooled vs Separate ---")

pooled_models_fresh = get_base_models()
for name, model in pooled_models_fresh.items():
    model.fit(X_train, y_train)
    y_pred_pooled = model.predict(X_test)

    # Separate prediction
    factory = model_factories[name]
    sep_metrics, sep_models = train_separate_strategy(
        factory, X_train, y_train, disaster_train,
        X_test, y_test, disaster_test
    )
    # Recalculer la prédiction séparée
    y_preds_sep = []
    for d, m in sep_models.items():
        y_preds_sep.append(m.predict(X_test))
    y_pred_sep = np.round(np.mean(y_preds_sep, axis=0)).astype(int) if y_preds_sep else np.zeros(len(y_test))

    stat, p_val = mcnemar_test(y_test.values, y_pred_pooled, y_pred_sep)
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
    print(f"{name}: McNemar stat={stat:.4f}, p={p_val:.4f} [{sig}]")

---
## 7. Importance des Caractéristiques

### 4 méthodes comparées :
1. **Permutation importance** : Mesure la dégradation quand on permute une feature. Agnostique au modèle.
2. **Coefficients (Logistic Regression)** : L'effet direct de chaque feature. Affecté par la multicolinéarité.
3. **Drop-column importance** : Mesure la dégradation quand on supprime une feature. Coûteux mais robuste.
4. **SHAP values** : Valeurs de Shapley. Théoriquement optimal, prend en compte les interactions.

In [ ]:
# Entraîner les modèles pour l'analyse d'importance
models_trained = get_base_models()
for name, model in models_trained.items():
    model.fit(X_train, y_train)

# 1. Permutation importance (sur Random Forest)
print("Calcul de la permutation importance...")
perm_imp = compute_permutation_importance(models_trained['Random Forest'], X_train, y_train)
print("\nPermutation Importance (Random Forest) :")
print(perm_imp.round(4).to_string())

In [ ]:
# 2. Coefficients de la Régression Logistique
lr_coefs = get_logistic_coefficients(models_trained['Logistic Regression'], feature_names)
print("Coefficients Logistic Regression (valeur absolue) :")
print(lr_coefs.round(4).to_string())

In [ ]:
# 3. Drop-column importance
print("Calcul de la drop-column importance (peut prendre du temps)...")
drop_imp = compute_drop_column_importance(
    lambda: RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42),
    X_train, y_train, cv=3
)
print("\nDrop-Column Importance :")
print(drop_imp.round(4).to_string())

In [ ]:
# 4. SHAP values
import shap

print("Calcul des SHAP values...")
explainer = shap.TreeExplainer(models_trained['XGBoost'])
shap_values = explainer.shap_values(X_train)

# SHAP summary plot
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_train, plot_type='bar', show=False)
plt.title('SHAP Feature Importance (XGBoost)', fontweight='bold')
plt.tight_layout()
plt.show()

# SHAP detailed summary
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_train, show=False)
plt.title('SHAP Values Détaillées (XGBoost)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Comparaison des 4 méthodes
shap_importance = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=feature_names
).sort_values(ascending=False)

importance_comparison = pd.DataFrame({
    'Permutation': perm_imp,
    'Coefficients (abs)': lr_coefs.abs() if len(lr_coefs) > 0 else np.nan,
    'Drop-Column': drop_imp,
    'SHAP (mean abs)': shap_importance
})

# Normaliser pour comparer les rangs
importance_ranks = importance_comparison.rank(ascending=False)

print("=== Comparaison des Rangs d'Importance ===")
print(importance_ranks.round(0).astype(int).to_string())

In [ ]:
# Visualisation comparative
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Comparaison des 4 Méthodes d\'Importance des Features', fontsize=14, fontweight='bold')

# Permutation
perm_imp.head(10).plot(kind='barh', ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title('1. Permutation Importance')
axes[0, 0].invert_yaxis()

# Coefficients
lr_coefs.abs().head(10).plot(kind='barh', ax=axes[0, 1], color='coral')
axes[0, 1].set_title('2. Coefficients LR (abs)')
axes[0, 1].invert_yaxis()

# Drop-column
drop_imp.head(10).plot(kind='barh', ax=axes[1, 0], color='mediumseagreen')
axes[1, 0].set_title('3. Drop-Column Importance')
axes[1, 0].invert_yaxis()

# SHAP
shap_importance.head(10).plot(kind='barh', ax=axes[1, 1], color='mediumpurple')
axes[1, 1].set_title('4. SHAP Mean |Value|')
axes[1, 1].invert_yaxis()

plt.tight_layout()
plt.show()

---
## 8. Discussion : Pourquoi les Méthodes ne Sont Pas d'Accord

### Sources de désaccord :

1. **Multicolinéarité** : Les features `sex_binary` et `female_high_class` sont corrélées. La régression logistique répartit le poids entre les deux, sous-estimant chacune. La permutation importance n'est pas affectée car elle mesure l'effet marginal.

2. **Non-linéarité** : Les coefficients LR capturent uniquement les effets linéaires. Si l'âge a un effet non-linéaire (U-shaped : très jeunes et très vieux ont plus de chances), LR le sous-estime. SHAP avec XGBoost capture cet effet.

3. **Interactions** : La drop-column importance ne capture pas les interactions : si `sex` et `social_class` sont redondantes, supprimer l'une ne dégrade pas le modèle car l'autre compense. SHAP gère correctement les interactions via les valeurs de Shapley.

4. **Variance d'estimation** : La permutation importance a une variance élevée pour les features fortement corrélées. L'importance par permutation peut surestimer l'importance de features non-informatives mais corrélées avec des features informatives.

### Quelle méthode est la plus fiable ?

Pour ce problème spécifique, **SHAP est la plus fiable** car :
- Notre dataset contient des features corrélées (multicolinéarité)
- Les relations ne sont pas linéaires
- Les interactions entre features sont importantes
- SHAP a des garanties théoriques (consistance, localité, efficience)

### Cas où la sélection automatique de features nuirait :

Si on utilise une sélection automatique basée sur la permutation importance, on risque de supprimer `social_class` car elle est corrélée avec `fare` et d'autres variables. Mais `social_class` est le concept le plus interprétable et le plus stable entre les catastrophes. Sa suppression dégraderait la généralisabilité sans forcément dégrader la performance in-sample.

---
## 9. Choix de la Stratégie pour le Déploiement

### Recommandation : Stratégie Pooled

Pour un scénario réel de prédiction sur une nouvelle catastrophe inconnue, la stratégie **pooled** est préférable car :

1. **Pas de connaissance a priori** : On ne sait pas à quelle catastrophe d'entraînement la nouvelle ressemblera
2. **Plus de données** : Le modèle pooled utilise toutes les données disponibles
3. **Patterns universels** : Certains facteurs (genre, âge, groupe) sont universels
4. **Simplicité** : Un seul modèle à maintenir et déployer

Cependant, si on pouvait caractériser la nouvelle catastrophe (type, époque, contexte), la stratégie **séparée** permettrait de pondérer les modèles en fonction de la similarité.